# Stochastic Programming

**Modern Operations Research with Python**

This chapter introduces optimization under uncertainty using **Stochastic Programming**. 
We focus on the practical **Two-Stage Stochastic Program with Recourse** implemented via Sample Average Approximation (SAA).

In [ ]:
import numpy as np
import pulp
import matplotlib.pyplot as plt

# Use GLPK (reliable on Apple M4)
solver = pulp.GLPK_CMD(msg=0)

np.random.seed(42)
print("✅ Libraries loaded – using GLPK solver")

## 6.1 Two-Stage Stochastic Newsvendor Problem

You must decide the order quantity **Q** (first-stage) **before** knowing the actual demand. 
After demand is revealed, you incur holding or shortage costs (second-stage recourse).

In [ ]:
# Problem parameters
c = 10.0   # unit ordering cost
p = 25.0   # unit selling price
h = 2.0    # unit holding cost for leftover inventory
b = 15.0   # unit shortage penalty

# Generate demand scenarios (Sample Average Approximation)
n_scenarios = 1000
demand_scenarios = np.random.normal(loc=150, scale=40, size=n_scenarios).clip(0)
prob = np.ones(n_scenarios) / n_scenarios

print(f"Generated {n_scenarios} demand scenarios. Mean demand: {demand_scenarios.mean():.1f}")

In [1]:
# Build the extensive form (deterministic equivalent)
prob_model = pulp.LpProblem("Two_Stage_Stochastic_Newsvendor", pulp.LpMinimize)

# First-stage decision
Q = pulp.LpVariable("Order_Quantity", lowBound=0)

# Second-stage variables (one per scenario)
sales = pulp.LpVariable.dicts("Sales", range(n_scenarios), lowBound=0)
leftover = pulp.LpVariable.dicts("Leftover", range(n_scenarios), lowBound=0)
shortage = pulp.LpVariable.dicts("Shortage", range(n_scenarios), lowBound=0)

# Objective: first-stage cost + expected second-stage cost
prob_model += c * Q + pulp.lpSum([
    prob[i] * (h * leftover[i] + b * shortage[i] - (p - c) * sales[i])
    for i in range(n_scenarios)
])

# Constraints for each scenario
for i in range(n_scenarios):
    d = demand_scenarios[i]
    prob_model += sales[i] <= Q
    prob_model += sales[i] <= d
    prob_model += leftover[i] == Q - sales[i]
    prob_model += shortage[i] == d - sales[i]

status = prob_model.solve(solver)

print("Solve status:", pulp.LpStatus[status])
print("Optimal first-stage order quantity Q:", pulp.value(Q))
print("Expected total cost:", pulp.value(prob_model.objective))

NameError: name 'pulp' is not defined

## 6.2 Visualization

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(demand_scenarios, bins=50, alpha=0.7, color='skyblue', label='Demand distribution')
plt.axvline(pulp.value(Q), color='red', linestyle='--', linewidth=2, label=f'Optimal Q = {pulp.value(Q):.1f}')
plt.xlabel('Demand')
plt.ylabel('Frequency')
plt.title('Two-Stage Stochastic Newsvendor — Optimal Order Quantity')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 6.3 Key Takeaways

- Stochastic programming explicitly accounts for uncertainty.
- Sample Average Approximation converts the stochastic problem into a large deterministic LP.
- For very large numbers of scenarios, decomposition algorithms (L-shaped, Benders) are needed.

**Next Chapter (Chapter 7)**: Hybrid OR-ML Approaches — using machine learning predictions inside optimization models.